# Stacking with time-ordered data

`StackingCVClassifier` and `StackingCVRegressor` accept `TimeSeriesSplit` without a new constructor option. The earliest observations can train base estimators even though they never appear in a test fold. Only observations with out-of-fold predictions train the meta-estimator.

This example uses synthetic data and one CPU worker. Keep observations in chronological order and compute lagged features using information available at prediction time. A cross-validation splitter cannot repair features that already contain future information.

## Regression and retained rows

The gap excludes observations immediately before each test fold. Each fold gets a fresh estimator clone, and final base estimators are refitted on all supplied training observations. That final refit is for subsequent prediction, not for generating the out-of-fold meta-training features.


In [1]:
import warnings
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import TimeSeriesSplit
from mlxtend.regressor import StackingCVRegressor

time = np.arange(24, dtype=float)
X = np.column_stack((time, np.sin(time / 3)))
y = 0.5 * time + np.cos(time / 4)
weights = np.linspace(1.0, 2.0, len(time))
cv = TimeSeriesSplit(n_splits=3, gap=2, max_train_size=10)
stack = StackingCVRegressor(
    regressors=[LinearRegression()], meta_regressor=Ridge(alpha=1.0),
    cv=cv, n_jobs=1, use_features_in_secondary=True,
    store_train_meta_features=True,
)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    stack.fit(X, y, sample_weight=weights)
assert any("Only samples" in str(w.message) for w in caught)
print(stack.train_meta_indices_)
print(stack.train_meta_features_.shape)
assert np.array_equal(stack.train_meta_indices_, np.arange(6, 24))
assert stack.train_meta_features_.shape == (18, 1)


[ 6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23]
(18, 1)


`train_meta_indices_` contains original **positional** row indices, sorted in ascending order. It is not a pandas index label. `train_meta_features_[j]` belongs to input row `train_meta_indices_[j]`. With `use_features_in_secondary=True`, the estimator adds those same rows of `X` for meta-training; the stored meta-feature array itself still contains only base predictions.

Targets and sample weights use the same row selection for meta-training. Each base fold receives weights for its training rows, while final base refits receive all weights. A warning announces reduced meta-training coverage; the example captures and checks it only to keep notebook output concise. Ordinary partitioning CV uses every row and does not emit that warning.

## Classification and missing fold classes

Probability columns use the complete training class vocabulary. A class absent from an early fold's training set gets a zero probability column for that fold. The chosen base estimator must itself support the classes present in each training fold; for example, a single-class fold is not valid training input for every classifier.


In [2]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from mlxtend.classifier import StackingCVClassifier

labels = np.array(["early"] * 8 + ["middle"] * 8 + ["late"] * 8)
classifier = StackingCVClassifier(
    classifiers=[DummyClassifier(strategy="prior")],
    meta_classifier=LogisticRegression(),
    cv=TimeSeriesSplit(n_splits=3), n_jobs=1,
    use_probas=True, store_train_meta_features=True,
)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    classifier.fit(X, labels)
assert any("Only samples" in str(w.message) for w in caught)
print(classifier.classes_)
print(classifier.train_meta_features_[0])
assert classifier.train_meta_features_.shape == (18, 3)
assert np.array_equal(classifier.train_meta_features_[0], [1.0, 0.0, 0.0])


['early' 'late' 'middle']
[1. 0. 0.]


## Scope and validation

- Direct partial-split support covers `TimeSeriesSplit` and explicit iterables of `(train_indices, test_indices)`. For a custom partial splitter, materialize its splits using `list(custom_cv.split(X, y, groups))` and pass that list as `cv`. This preserves ordinary splitter objects' existing execution and random-state behavior.
- Test folds must not overlap; repeated test coverage is not averaged. Indices must be nonempty, one-dimensional, nonnegative integer positions within the input. A fold's training and test sets must not overlap. Explicit split iterables remain responsible for chronological and group isolation.
- Partial classification supports a single output. Every target class must appear somewhere among retained meta-training rows; otherwise fitting raises an error. This prevents `classes_` from disagreeing with meta-estimator probability outputs. Multi-output regression is supported.
- The partial path rejects enabled scikit-learn metadata routing. It supports the existing explicit `sample_weight` argument without routing. Rejecting routing avoids silently ignoring estimator metadata requests. Ordinary CV retains its existing scikit-learn routing behavior.
- `drop_proba_col` is applied after missing-class probability alignment. `use_clones`/`refit` retain their existing final-estimator semantics; independent fold estimators are still cloned.
- Full-data base refits do not make training-set predictions an unbiased evaluation. Evaluate the final stack on a separate future holdout or an appropriate outer time-series split.

Related discussions: [classifier #497](https://github.com/rasbt/mlxtend/issues/497) and [regressor #1101](https://github.com/rasbt/mlxtend/issues/1101).
